In [28]:
# STEP 1 — Preprocessing + Feature Engineering with Auxiliary Data

import re
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import BallTree

DATA_DIR = "."
TRAIN_PATH = f"{DATA_DIR}/data/train.csv"
TEST_PATH  = f"{DATA_DIR}/data/test.csv"
HDB_PATH   = f"{DATA_DIR}/data/sg-hdb.csv"

# auxiliary
MRT_PATH   = f"{DATA_DIR}/data/auxiliary-data/sg-mrt-stations.csv"
PRI_PATH   = f"{DATA_DIR}/data/auxiliary-data/sg-primary-schools.csv"
SEC_PATH   = f"{DATA_DIR}/data/auxiliary-data/sg-secondary-schools.csv"
MALL_PATH  = f"{DATA_DIR}/data/auxiliary-data/sg-shopping-malls.csv"

# ---------- load ----------
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
hdb   = pd.read_csv(HDB_PATH)

mrt   = pd.read_csv(MRT_PATH)
pri   = pd.read_csv(PRI_PATH)
sec   = pd.read_csv(SEC_PATH)
mall  = pd.read_csv(MALL_PATH)

EARTH_R = 6371000.0  # meters

# ---------- normalize area ----------
def normalize_area_name(name):
    name = str(name).lower().strip()
    mapping = {
        'jurong west': 'jurongwest', 'jurong east': 'jurongeast',
        'bukit timah': 'bukittimah', 'bukit batok': 'bukitbatok',
        'bukit panjang': 'bukitpanjang', 'bukit merah': 'bukitmerah',
        'toa payoh': 'toapayoh', 'pasir ris': 'pasiris',
        'ang mo kio': 'angmokio', 'choa chu kang': 'choachukang',
    }
    return mapping.get(name, name.replace(' ', ''))

def normalize_area_columns(df):
    for col in ["PLN_AREA","PLANNING_AREA","TOWN"]:
        if col in df.columns:
            df["AREA"] = df[col].astype(str).str.strip().apply(normalize_area_name)
            break
    return df

for dataset in [hdb,mrt,pri,sec,mall,train,test]:
    dataset = normalize_area_columns(dataset)

# ---------- unify lease column ----------
def normalize_columns(df):
    if "LEASE_COMMENCE_DATA" in df.columns and "LEASE_COMMENCE_DATE" not in df.columns:
        df = df.rename(columns={"LEASE_COMMENCE_DATA":"LEASE_COMMENCE_DATE"})
    return df

train = normalize_columns(train)
test  = normalize_columns(test)

# ---------- text cleaning ----------
def to_lower_strip(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip().str.lower()
    return df

text_cols = ["TOWN","FLAT_TYPE","BLOCK","STREET","FLOOR_RANGE","FLAT_MODEL","ECO_CATEGORY"]
train = to_lower_strip(train, text_cols)
test  = to_lower_strip(test,  text_cols)

# canonicalize FLAT_TYPE
canon_map = {
    "1 room":"1-room","2 room":"2-room","3 room":"3-room",
    "4 room":"4-room","5 room":"5-room",
    "1-room":"1-room","2-room":"2-room","3-room":"3-room",
    "4-room":"4-room","5-room":"5-room",
    "multi generation":"multi generation","executive":"executive"
}
for df in (train,test):
    if "FLAT_TYPE" in df.columns:
        df["FLAT_TYPE"] = df["FLAT_TYPE"].map(lambda x: canon_map.get(x,x))

# ---------- time features ----------
def add_time_features(df):
    dt = pd.to_datetime(df["MONTH"], format="%Y-%m", errors="coerce")
    df["SALE_YEAR"]  = dt.dt.year
    df["SALE_MONTH"] = dt.dt.month
    base = dt.min()
    df["MONTH_IDX"] = ((dt.dt.year-base.year)*12 + (dt.dt.month-base.month)).astype("Int64")
    return df
train = add_time_features(train)
test  = add_time_features(test)

# ---------- floor features ----------
def parse_floor_range(s):
    m = re.match(r"^\s*(\d{1,2})\s*to\s*(\d{1,2})\s*$", str(s))
    if not m: return np.nan,np.nan,np.nan
    lo,hi = int(m.group(1)),int(m.group(2))
    return lo,hi,(lo+hi)/2.0

for df in (train,test):
    lo_hi_med = df["FLOOR_RANGE"].apply(parse_floor_range)
    df["FLOOR_MIN"] = lo_hi_med.apply(lambda t:t[0])
    df["FLOOR_MAX"] = lo_hi_med.apply(lambda t:t[1])
    df["FLOOR_MEDIAN"] = lo_hi_med.apply(lambda t:t[2])

# ---------- lease/age ----------
def add_lease_features(df):
    yr = df["LEASE_COMMENCE_DATE"].astype("Int64")
    df["FLAT_AGE"] = df["SALE_YEAR"] - yr
    df["REMAINING_LEASE"] = 99 - df["FLAT_AGE"]
    return df
train = add_lease_features(train)
test  = add_lease_features(test)

# ---------- flat type ----------
def rooms_from_type(s):
    m = re.match(r"^(\d+)-room$", str(s))
    return int(m.group(1)) if m else np.nan

for df in (train,test):
    df["ROOMS"] = df["FLAT_TYPE"].apply(rooms_from_type)
    df["IS_EXECUTIVE"] = (df["FLAT_TYPE"]=="executive").astype(int)
    df["AREA_PER_ROOM"] = df["FLOOR_AREA_SQM"] / np.where(
        pd.notna(df["ROOMS"])&(df["ROOMS"]>0), df["ROOMS"], np.nan)

# ---------- block parsing ----------
def split_block(b):
    m = re.match(r"^(\d+)([a-z])?$", str(b))
    if not m: return np.nan,0
    return int(m.group(1)),1 if m.group(2) else 0

for df in (train,test):
    vals = df["BLOCK"].apply(split_block)
    df["BLOCK_NUM"] = vals.apply(lambda t:t[0])
    df["BLOCK_HAS_SUFFIX"] = vals.apply(lambda t:t[1])

# ---------- street short-name normalizer ----------
street_map={
    # suffixes
    "street":"st","road":"rd","avenue":"ave","drive":"dr","crescent":"cres",
    "lane":"ln","place":"pl","close":"cl","terrace":"ter","square":"sq",
    "walk":"walk","view":"view","grove":"grove","park":"pk","hill":"hl",
    "link":"lk","rise":"rise","green":"gr","heights":"hts","gardens":"gdns",
    "garden":"gdn","boulevard":"blvd","court":"ct","centre":"ctr",
    "central":"ctrl","estate":"est","market":"mkt","expressway":"expy",
    "square":"sq","townhouse":"townhse","complex":"cplx","condominium":"condo",
    "carpark":"cp","factory":"fty","library":"lib","playground":"p/g",
    "station":"stn","point":"pt",
    # prefixes
    "bukit":"bt","lorong":"lor","jalan":"jln","tanjong":"tg","mount":"mt",
    "kampong":"kg","north":"nth","south":"sth","upper":"upp","saint":"st.",
    # misc terms
    "assembly":"ably","administration":"admin","apartment":"apt",
    "apartments":"apts","ayer rajah expressway":"aye",
    "bukit timah expressway":"bke","branch office":"bo","branch":"br",
    "buddhist":"budd","cathedral":"cath","central business district":"cbd",
    "community centre":"cc","church":"ch","chambers":"chbrs","cinema":"cine",
    "cinemas":"cines","clubhouse":"clubhse","commonwealth":"c'wealth",
    "department":"dept","development":"devt","division":"div","education":"edn",
    "engineering":"engrg","environment":"env",
    "electronic road pricing":"erp","east coast expressway":"ecp",
    "headquarter":"hqr","headquarters":"hqrs","historic site":"hs",
    "hospital":"hosp","institute":"inst","institution":"instn",
    "international":"intl","junior colleges":"jc","junior":"jnr",
    "kranji expressway":"kje","kilometre":"km",
    "kallang paya lebar expressway":"kpe","maisonette":"mai",
    "maisonettes":"mais","mansion":"man","mansions":"mans",
    "marina coastal expressway":"mce","metropolitan":"met","methodist":"meth",
    "ministry":"min","masjid":"mjd","national":"natl",
    "neighbourhood police centres":"npc",
    "neighbourhood police posts":"npp","open space":"o/s","pulau":"p",
    "pan island expressway":"pie","polyclinic":"poly","presbyterian":"presby",
    "redevelopment":"redevt","sungei":"s","seletar expressway":"sle",
    "singapore":"s'pore","tampines expressway":"tpe",
    "town council":"tc","technical":"tech","under construction":"u/c",
    "vocational":"voc","warehouse":"warehse",
    # housing / buildings
    "block":"blk","building":"bldg","house":"hse","school":"sch",
    "primary":"pri","secondary":"sec","college":"jc","industrial":"ind"
}



def normalize_street(s: str) -> str:
    s = str(s).lower().strip()
    tokens = s.split()
    tokens = [street_map.get(tok, tok) for tok in tokens]
    return " ".join(tokens)


# apply keys
for df in (train, test, hdb):
    df["BLOCK_KEY"]  = df["BLOCK"].astype(str).str.lower().str.strip()
    df["STREET_KEY"] = df["STREET"].astype(str).apply(normalize_street)

# ---------- add coordinates from sg-hdb ----------
def add_coords(df, hdb):
    df = df.copy()
    merged = df.merge(
        hdb[["BLOCK_KEY","STREET_KEY","LATITUDE","LONGITUDE"]],
        on=["BLOCK_KEY","STREET_KEY"], how="left"
    )
    return merged

train = add_coords(train, hdb)
test  = add_coords(test, hdb)

# ---------- debug: check missing coords ----------
missing_train = train[pd.isna(train["LATITUDE"])]
missing_test  = test[pd.isna(test["LATITUDE"])]

print("Train coords merged:", train[["BLOCK","STREET","LATITUDE","LONGITUDE"]].head())
print("Test coords merged:", test[["BLOCK","STREET","LATITUDE","LONGITUDE"]].head())
print("Missing in train:", len(missing_train), "/", len(train))
print("Missing in test :", len(missing_test), "/", len(test))

if len(missing_train) > 0:
    print("\nSample missing BLOCK+STREET in train:")
    print(missing_train[["BLOCK","STREET"]].head(20))

# ---------- area-restricted distances ----------
def nearest_area_distance(df, fac_df, key):
    out = np.full(len(df), np.nan, dtype=float)
    for area, grp in df.groupby("AREA"):
        facs = fac_df[fac_df["AREA"] == area]
        if facs.empty:
            continue
        # only keep rows with valid coordinates
        grp_valid = grp[pd.notna(grp["LATITUDE"]) & pd.notna(grp["LONGITUDE"])]
        if grp_valid.empty:
            continue
        # build tree from facilities
        tree = BallTree(np.radians(facs[["LATITUDE","LONGITUDE"]]), metric="haversine")
        q = np.radians(grp_valid[["LATITUDE","LONGITUDE"]].to_numpy(dtype=float))
        d_rad, _ = tree.query(q, k=1)
        d_m = d_rad[:, 0] * EARTH_R
        out[grp_valid.index] = d_m
    df[f"DIST_{key}"] = out
    df[f"DIST_{key}_MISSING"] = pd.isna(out).astype(int)
    df[f"DIST_{key}"] = df[f"DIST_{key}"].fillna(1e6)
    return df


facilities = {"MRT":mrt,"PRI":pri,"SEC":sec,"MALL":mall}
for key,tbl in facilities.items():
    train = nearest_area_distance(train,tbl,key)
    test  = nearest_area_distance(test,tbl,key)

# ---------- encodings ----------
for col in ["STREET","BLOCK"]:
    full = pd.concat([train[col],test[col]])
    counts = full.value_counts().astype(int)
    train[f"{col}_COUNT"]=train[col].map(counts)
    test[f"{col}_COUNT"]=test[col].map(counts)

if "ECO_CATEGORY" in train.columns and train["ECO_CATEGORY"].nunique(dropna=False)<=1:
    train=train.drop(columns=["ECO_CATEGORY"])
    test=test.drop(columns=["ECO_CATEGORY"])

cat_cols=["TOWN","FLAT_TYPE","FLAT_MODEL"]
for col in cat_cols:
    if col in train.columns:
        le=LabelEncoder()
        full=pd.concat([train[col],test[col]]).astype(str).fillna("nan")
        le.fit(full)
        train[col]=le.transform(train[col].astype(str).fillna("nan"))
        test[col]=le.transform(test[col].astype(str).fillna("nan"))

# ---------- final features ----------
target_col="RESALE_PRICE"
base_features=[
    "FLOOR_AREA_SQM","FLOOR_MIN","FLOOR_MAX","FLOOR_MEDIAN",
    "SALE_YEAR","SALE_MONTH","MONTH_IDX",
    "FLAT_AGE","REMAINING_LEASE",
    "ROOMS","IS_EXECUTIVE","AREA_PER_ROOM",
    "BLOCK_NUM","BLOCK_HAS_SUFFIX","STREET_COUNT","BLOCK_COUNT",
    "TOWN","FLAT_TYPE","FLAT_MODEL",
    "DIST_MRT","DIST_PRI","DIST_SEC","DIST_MALL",
    "DIST_MRT_MISSING","DIST_PRI_MISSING","DIST_SEC_MISSING","DIST_MALL_MISSING"
]
features=[c for c in base_features if c in train.columns]

X=train[features].copy(); y=train[target_col].copy()
X_test=test[features].copy()

print("Train shape:",X.shape," Test shape:",X_test.shape)
print("Features:",features[:8],"...",len(features),"total")

X.join(y).to_csv("train_processed_extra.csv",index=False)
X_test.to_csv("test_processed_extra.csv",index=False)


Train coords merged:   BLOCK              STREET  LATITUDE   LONGITUDE
0  681b  woodlands drive 62  1.439325  103.803324
1   264    bishan street 24  1.358668  103.842070
2   520       jelapang road  1.386468  103.766508
3  121b     edgedale plains  1.393089  103.909042
4  997b   buangkok crescent  1.385902  103.881387
Test coords merged:   BLOCK            STREET  LATITUDE   LONGITUDE
0  115d     canberra walk  1.445544  103.828992
1  118a     jalan membina  1.281420  103.825645
2    22     sin ming road  1.357284  103.839397
3   635  hougang avenue 8  1.370212  103.879116
4   275      bangkit road  1.376836  103.774826
Missing in train: 1 / 162691
Missing in test : 0 / 50000

Sample missing BLOCK+STREET in train:
      BLOCK          STREET
35901   42a  margaret drive
Train shape: (162691, 27)  Test shape: (50000, 27)
Features: ['FLOOR_AREA_SQM', 'FLOOR_MIN', 'FLOOR_MAX', 'FLOOR_MEDIAN', 'SALE_YEAR', 'SALE_MONTH', 'MONTH_IDX', 'FLAT_AGE'] ... 27 total


In [29]:
import pandas as pd

# === Load preprocessed datasets ===
train = pd.read_csv("train_processed_extra.csv")
test  = pd.read_csv("test_processed_extra.csv")

# === Handle missing distance features ===
distance_cols = ["DIST_MRT", "DIST_PRI", "DIST_SEC", "DIST_MALL"]

for col in distance_cols:
    # Add indicator column (1 = missing, 0 = present)
    train[f"{col}_MISSING"] = train[col].isna().astype(int)
    test[f"{col}_MISSING"] = test[col].isna().astype(int)

    # Fill missing with a large sentinel (1e6 meters = effectively "very far")
    train[col] = train[col].fillna(1e6)
    test[col] = test[col].fillna(1e6)

# === Save updated datasets ===
train.to_csv("train_processed_extra.csv", index=False)
test.to_csv("test_processed_extra.csv", index=False)

# === Report missing counts ===
print("=== Train Missing Summary ===")
train_missing_summary = train[[c+"_MISSING" for c in distance_cols]].sum()
print(train_missing_summary)
print((train_missing_summary / len(train) * 100).round(2).astype(str) + "%")

print("\n=== Test Missing Summary ===")
test_missing_summary = test[[c+"_MISSING" for c in distance_cols]].sum()
print(test_missing_summary)
print((test_missing_summary / len(test) * 100).round(2).astype(str) + "%")

print("\nSample with flags:")
print(train[distance_cols + [c+"_MISSING" for c in distance_cols]].head())


=== Train Missing Summary ===
DIST_MRT_MISSING     0
DIST_PRI_MISSING     0
DIST_SEC_MISSING     0
DIST_MALL_MISSING    0
dtype: int64
DIST_MRT_MISSING     0.0%
DIST_PRI_MISSING     0.0%
DIST_SEC_MISSING     0.0%
DIST_MALL_MISSING    0.0%
dtype: object

=== Test Missing Summary ===
DIST_MRT_MISSING     0
DIST_PRI_MISSING     0
DIST_SEC_MISSING     0
DIST_MALL_MISSING    0
dtype: int64
DIST_MRT_MISSING     0.0%
DIST_PRI_MISSING     0.0%
DIST_SEC_MISSING     0.0%
DIST_MALL_MISSING    0.0%
dtype: object

Sample with flags:
      DIST_MRT    DIST_PRI    DIST_SEC    DIST_MALL  DIST_MRT_MISSING  \
0   306.038576  168.454975  268.500127  1994.705323                 0   
1  1002.079136  532.048296  405.667355  1175.546836                 0   
2  2667.355366  156.801868  238.400764  2993.494835                 0   
3   872.326535  177.342222  522.130026   463.176435                 0   
4  2013.650343  949.539689  927.449928  2029.523019                 0   

   DIST_PRI_MISSING  DIST_SEC_MISSI

In [30]:
# STEP 2 — XGBoost training with 5-Fold CV

import pandas as pd
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np

# === Load processed data ===
train = pd.read_csv("train_processed_extra.csv")
test  = pd.read_csv("test_processed_extra.csv")

# target
y = train["RESALE_PRICE"]

# features: all except target
feature_cols = [c for c in train.columns if c != "RESALE_PRICE"]
X = train[feature_cols]
X_test = test[feature_cols]

# DMatrix format for efficiency
Xy = xgb.DMatrix(X, label=y)
X_test_dm = xgb.DMatrix(X_test)

# model hyperparameters
params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "learning_rate": 0.05,
    "max_depth": 7,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "seed": 42,
}

# 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    print(f"\n=== Fold {fold+1} ===")
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dvalid = xgb.DMatrix(X_val, label=y_val)

    model = xgb.train(
        params,
        dtrain,
        num_boost_round=2000,
        evals=[(dtrain, "train"), (dvalid, "valid")],
        early_stopping_rounds=100,
        verbose_eval=100,
    )

    oof_preds[val_idx] = model.predict(dvalid, iteration_range=(0, model.best_iteration))
    test_preds += model.predict(X_test_dm, iteration_range=(0, model.best_iteration)) / kf.n_splits

# overall RMSE
rmse = mean_squared_error(y, oof_preds, squared=False)
print(f"\nOverall CV RMSE: {rmse:.4f}")

# save predictions
np.save("oof_preds.npy", oof_preds)
np.save("test_preds.npy", test_preds)



=== Fold 1 ===
[0]	train-rmse:523451.27973	valid-rmse:523786.80659
[100]	train-rmse:42148.05663	valid-rmse:44181.32813
[200]	train-rmse:31902.20336	valid-rmse:34333.41005
[300]	train-rmse:28346.25922	valid-rmse:31090.99297
[400]	train-rmse:26444.30343	valid-rmse:29487.26912
[500]	train-rmse:25105.42906	valid-rmse:28453.87407
[600]	train-rmse:24202.80282	valid-rmse:27809.99568
[700]	train-rmse:23491.20782	valid-rmse:27387.81261
[800]	train-rmse:22870.98384	valid-rmse:27056.40629
[900]	train-rmse:22331.62615	valid-rmse:26783.90155
[1000]	train-rmse:21858.06359	valid-rmse:26588.09029
[1100]	train-rmse:21454.40222	valid-rmse:26425.66241
[1200]	train-rmse:21063.32745	valid-rmse:26294.98605
[1300]	train-rmse:20726.18118	valid-rmse:26207.25647
[1400]	train-rmse:20397.60780	valid-rmse:26121.05290
[1500]	train-rmse:20092.72332	valid-rmse:26044.48522
[1600]	train-rmse:19815.64396	valid-rmse:25988.95510
[1700]	train-rmse:19548.86972	valid-rmse:25929.88527
[1800]	train-rmse:19287.67067	valid-rmse

In [ ]:
# STEP 3 — Create Kaggle submission file

import pandas as pd
import numpy as np

# load example submission to get correct format
sub = pd.read_csv("data/example-submission.csv")

# load test predictions from Step 2
test_preds = np.load("test_preds.npy")

# assign predictions to the second column (whatever it's named)
target_col = sub.columns[1]
sub[target_col] = test_preds

# save with your custom filename
sub.to_csv("submission_aodi.csv", index=False)

print("submission_aodi.csv created, shape:", sub.shape)
print(sub.head())


submission_aodi3.csv created, shape: (50000, 2)
   Id     Predicted
0   0  532398.12500
1   1  571792.93750
2   2  380368.93750
3   3  441111.84375
4   4  523267.53125


In [ ]:
# STEP 4 — XGBoost with Bayesian Optimization
import pandas as pd
import numpy as np
import xgboost as xgb
import sys
try:
    import importlib.metadata as importlib_metadata  # Python 3.8+
except ImportError:
    import importlib_metadata as importlib_metadata  # backport for <3.8
sys.modules['importlib.metadata'] = importlib_metadata
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from skopt import gp_minimize
from skopt.space import Integer, Real
from skopt.utils import use_named_args


# === Load data ===
train = pd.read_csv("train_processed_extra.csv")
test  = pd.read_csv("test_processed_extra.csv")

y = train["RESALE_PRICE"]
feature_cols = [c for c in train.columns if c != "RESALE_PRICE"]
X = train[feature_cols]
X_test = test[feature_cols]

# search space
space = [
    Integer(6, 10, name="max_depth"),
    Real(0.01, 0.1, prior="log-uniform", name="learning_rate"),
    Integer(1, 10, name="min_child_weight"),
    Real(0.5, 1.0, name="subsample"),
    Real(0.5, 1.0, name="colsample_bytree"),
]

# base params
base_params = {
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "seed": 42,
}

# evaluation function
@use_named_args(space)
def objective(**params):
    full_params = base_params.copy()
    full_params.update(params)

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X))

    for tr_idx, val_idx in kf.split(X):
        dtrain = xgb.DMatrix(X.iloc[tr_idx], label=y.iloc[tr_idx])
        dvalid = xgb.DMatrix(X.iloc[val_idx], label=y.iloc[val_idx])
        model = xgb.train(
            full_params, dtrain,
            num_boost_round=2000,
            evals=[(dvalid, "valid")],
            early_stopping_rounds=100,
            verbose_eval=False,
        )
        oof_preds[val_idx] = model.predict(
            dvalid, iteration_range=(0, model.best_iteration)
        )

    rmse = mean_squared_error(y, oof_preds, squared=False)
    print(f"Params: {params} → CV RMSE {rmse:.4f}")
    return rmse

# run Bayesian optimization
res = gp_minimize(
    objective,
    space,
    n_calls=30,       # number of trials
    n_random_starts=5,# initial random points
    random_state=42,
)

print("\nBest RMSE:", res.fun)
print("Best params:", res.x)

# retrain with best params
best_params = base_params.copy()
best_params.update(dict(zip([s.name for s in space], res.x)))

dtrain_full = xgb.DMatrix(X, label=y)
dtest = xgb.DMatrix(X_test)

best_model = xgb.train(
    best_params,
    dtrain_full,
    num_boost_round=2000,
    evals=[(dtrain_full, "train")],
    early_stopping_rounds=100,
    verbose_eval=100,
)

test_preds = best_model.predict(dtest, iteration_range=(0, best_model.best_iteration))

# save
np.save("test_preds.npy", test_preds)
pd.DataFrame({"Predicted": test_preds}).to_csv("test_preds_bayes.csv", index=False)

print("Saved: test_preds.npy & test_preds_bayes.csv")


Params: {'max_depth': 9, 'learning_rate': 0.015255793090045737, 'min_child_weight': 8, 'subsample': 0.7984250789732436, 'colsample_bytree': 0.7229163764267956} → CV RMSE 25893.2947
Params: {'max_depth': 6, 'learning_rate': 0.028790479097932958, 'min_child_weight': 4, 'subsample': 0.5714334089609704, 'colsample_bytree': 0.8254442364744266} → CV RMSE 27091.5823
Params: {'max_depth': 6, 'learning_rate': 0.05272283709671439, 'min_child_weight': 9, 'subsample': 0.5003893829205072, 'colsample_bytree': 0.9961057796456089} → CV RMSE 26655.2422
Params: {'max_depth': 8, 'learning_rate': 0.04089339433965304, 'min_child_weight': 1, 'subsample': 0.5115312125207079, 'colsample_bytree': 0.7623873301291946} → CV RMSE 25820.3145
Params: {'max_depth': 8, 'learning_rate': 0.011134370364229916, 'min_child_weight': 10, 'subsample': 0.6163856702151521, 'colsample_bytree': 0.5453032172664104} → CV RMSE 26948.2443
Params: {'max_depth': 8, 'learning_rate': 0.05416758890068957, 'min_child_weight': 1, 'subsample